# Vertex AI RAG Engine + Gemini 3 Flash: Full Lifecycle

This notebook demonstrates the **complete lifecycle** of using the Vertex AI RAG Engine with LangChain:
1. **Management**: Create a RAG Corpus and import data using the Vertex AI SDK.
2. **Retrieval**: Connect the corpus to LangChain using `VertexRagRetriever`.
3. **Generation**: Query using **Gemini 3 Flash** on the **global** endpoint.

## 1. Install Dependencies

In [21]:
%pip install --upgrade -q langchain-google-vertexai google-cloud-aiplatform --quiet
!pip install -U langchain-google-genai --quiet

## 2. Authenticate and Initialize

In [10]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "upasanapati-srtt"  # @param {type:"string"}
LOCATION = "us-central1" # @param {type:"string"}

import vertexai
from vertexai import rag
from vertexai.generative_models import Tool

vertexai.init(project=PROJECT_ID, location=LOCATION)

## 3. Setup: Create Corpus and Ingest Data

We use the direct `vertexai.rag` SDK for management tasks.

In [12]:
# 1. Configure the embedding model
embedding_model_config = rag.RagEmbeddingModelConfig(
    vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
        publisher_model="publishers/google/models/text-embedding-004"
    )
)

# 2. Create the Corpus
rag_corpus = rag.create_corpus(
    display_name="my_langchain_corpus",
    backend_config=rag.RagVectorDbConfig(
        rag_embedding_model_config=embedding_model_config
    ),
)

# 3. Import Files (Supports GCS and Google Drive)
rag.import_files(
    rag_corpus.name,
    paths=["gs://cloud-samples-data/generative-ai/pdf/"], # Example public sample data
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(chunk_size=512, chunk_overlap=100)
    )
)

print(f"Created corpus: {rag_corpus.name}")

Created corpus: projects/1019100070692/locations/us-central1/ragCorpora/9144559043375792128


## 4. Integration: LangChain Retrieval Chain

Now we plug the managed corpus into a standard LangChain flow.

In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from vertexai import rag

# 1. Custom Retrieval Function (Remains the same as it uses the Vertex AI SDK directly)
def retrieve_from_corpus(query):
    response = rag.retrieval_query(
        rag_resources=[rag.RagResource(rag_corpus=rag_corpus.name)],
        text=query,
        rag_retrieval_config=rag.RagRetrievalConfig(top_k=3),
    )
    return "\n\n".join([c.text for c in response.contexts.contexts])

# 2. Setup Gemini 2.5 Pro using the latest class
PROJECT_ID = "upasanapati-srtt"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    project=PROJECT_ID,   # <--- Added explicit project ID
    location=LOCATION,
    vertexai=True         # <--- Essential to specify you are using the Vertex AI backend
)

# 3. Run a Query
query = "What is this document about?"
context = retrieve_from_corpus(query)

# Define prompt
system_prompt = "Use the retrieved context to answer correctly.\n\n{context}"
prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", "{input}")])

# Create and run chain
chain = prompt | llm
response = chain.invoke({"input": query, "context": context})

print(f"Answer: {response.content}")

Answer: Based on the provided context, this document is an **exhibit list** from a regulatory filing made by **Johnson & Johnson** to the U.S. Securities and Exchange Commission (SEC).

Specifically, it is an index that lists and describes all the supporting documents that are either included with the main report or have been previously filed and are being incorporated by reference.

The exhibits listed include:
*   Corporate governance documents (Certificate of Incorporation, By-Laws).
*   Executive compensation plans (Long-Term Incentive Plans, Severance Pay Plan).
*   Certifications by the Chief Executive Officer (CEO) and Chief Financial Officer (CFO) as required by the Sarbanes-Oxley Act.
*   A list of the company's subsidiaries.
*   Consent from the company's accounting firm.
*   Financial data files in XBRL format.
